# Fine-Tuning LoRA - Modèle Médical Expérimental
## TechCorp Industries - Équipe IA/DATA

Ce notebook effectue le fine-tuning LoRA d'un modèle Phi-3.5-mini-instruct sur un dataset médical.

**Prérequis:** GPU activé dans Colab (Runtime > Change runtime type > T4 GPU)

In [ ]:
# Installation des dépendances
!pip install -q unsloth transformers datasets peft trl accelerate bitsandbytes
!pip install -q huggingface_hub

In [ ]:
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 1. Téléchargement et préparation du dataset médical

In [ ]:
from datasets import load_dataset
import json

print("Téléchargement du dataset médical...")
dataset = load_dataset("ruslanmv/ai-medical-chatbot")
print(f"Dataset: {dataset}")
print(f"Colonnes: {dataset['train'].column_names}")
print(f"\nExemple:")
print(json.dumps(dataset['train'][0], indent=2, ensure_ascii=False))

In [ ]:
import re

SYSTEM_PROMPT = (
    "Tu es un assistant médical expérimental. Tu fournis des informations médicales "
    "générales à titre éducatif uniquement. Tu ne remplaces pas un avis médical professionnel."
)

def clean_text(text):
    if not isinstance(text, str):
        return str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'<[^>]+>', '', text)
    return text

# Détection automatique des champs
sample = dataset['train'][0]
q_candidates = ['Patient', 'Question', 'question', 'input', 'instruction', 'Description']
a_candidates = ['Doctor', 'Answer', 'answer', 'output', 'response']

q_field = next((f for f in q_candidates if f in sample), None)
a_field = next((f for f in a_candidates if f in sample), None)
print(f"Champs détectés - Question: '{q_field}', Réponse: '{a_field}'")

def format_for_training(example):
    question = clean_text(example.get(q_field, ''))
    answer = clean_text(example.get(a_field, ''))
    text = (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{question}<|end|>\n"
        f"<|assistant|>\n{answer}<|end|>"
    )
    return {"text": text}

# Nettoyage et filtrage
def is_valid(example):
    q = str(example.get(q_field, '')).strip()
    a = str(example.get(a_field, '')).strip()
    return len(q) > 5 and len(a) > 10

filtered = dataset['train'].filter(is_valid)
formatted = filtered.map(format_for_training)
print(f"Échantillons après filtrage: {len(formatted)} / {len(dataset['train'])}")

# Split train/val
split = formatted.train_test_split(test_size=0.1, seed=42)
train_data = split['train']
val_data = split['test']
print(f"Train: {len(train_data)}, Val: {len(val_data)}")
print(f"\nExemple formaté:\n{train_data[0]['text'][:500]}")

## 2. Chargement du modèle de base + configuration LoRA

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Phi-3.5-mini-instruct"
MAX_SEQ_LENGTH = 2048

print(f"Chargement de {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
print("Modèle chargé!")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Paramètres entraînables: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 3. Fine-tuning

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=val_data,
    args=TrainingArguments(
        output_dir="./medical_lora_output",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        weight_decay=0.01,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=3,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        seed=42,
        report_to="none",
    ),
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
)

print("Démarrage du fine-tuning...")
result = trainer.train()
print(f"\nTerminé! Loss finale: {result.training_loss:.4f}")

## 4. Sauvegarde de l'adaptateur LoRA

In [ ]:
SAVE_PATH = "./medical_lora_adapter"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Adaptateur LoRA sauvegardé: {SAVE_PATH}")

# Télécharger les fichiers
import shutil
shutil.make_archive("medical_lora_adapter", 'zip', SAVE_PATH)
print("Archive créée: medical_lora_adapter.zip")

# Si sur Colab, télécharger automatiquement
try:
    from google.colab import files
    files.download('medical_lora_adapter.zip')
except ImportError:
    pass

## 5. Tests du modèle fine-tuné

In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = [
    "A patient presents with persistent headache, fever, and stiff neck. What are the possible diagnoses?",
    "What are the main side effects of metformin?",
    "Describe the emergency management protocol for anaphylactic shock.",
    "What are the recommended cancer screening guidelines for adults over 50?",
    "Can you prescribe me antibiotics for my cold?",
]

for prompt in test_prompts:
    formatted_prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{prompt}<|end|>\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"Q: {prompt}")
    print(f"R: {response[:500]}")

In [ ]:
print("\n" + "="*60)
print("FINE-TUNING ET TESTS TERMINÉS")
print("="*60)
print(f"Modèle de base: {MODEL_NAME}")
print(f"Adaptateur LoRA: {SAVE_PATH}")
print(f"Loss finale: {result.training_loss:.4f}")
print("\nN'oubliez pas de télécharger medical_lora_adapter.zip !")